**WEEK 3: DAY 1**

Welcome to **Google Colab** - using free and powerful compute in the cloud.

**Google Colab** gives you a **remote Notebook-style browser window** on to a machine, and you run code "locally" on that machine.

**Benefits of Colab**
1. Free access to **T4 GPUs**.
2. Easy ability to share code and collaborate on it.
3. Everyone gets to use identical code -- no environment differences.

**Downsides of Colab**

1. As it's free, Google reserves the right to bump you off the box at any point ("reset the runtime"), and this happens most quickly if no code is running.
2. They can also downgrade you from a T4 to a CPU-only box. Sometimes this happens silently. You need to start everything again from the top. Paid plans last longer.
3. You need to pip install packages every single time.
4. There's some latency - it's not as interactive as coding on your own box.

**Survival Guide**


1. Always start by pressing the drop down arrow by Connect on the top right, and "Connect to a hosted runtime: T4".
2. From that dropdown, "View Resources" to check you have a GPU and monitor memory.
3. If things go awry, Runtime >> Disconnect and Delete Runtime, Connect again, and then run cells from the top.
4. Always run the pip installs! Ignore pip dependency errors.
5. Runtime >> Restart session: this restarts the Python Kernel, but pip packages remain installed and the disk remains the same.
6.Runtime >> Disconnect and delete session. This wipes everything.

**transformers**==4.56.2: Installs version 4.56.2 of Hugging Face's Transformers library. This is the industry-standard library used to download, run, and fine-tune large models like BERT, GPT, Llama, and various Text-to-Speech/Speech-to-Text models.

**diffusers**==0.32.2: Installs version 0.32.2 of Hugging Face's Diffusers library. This library is specifically built for image and audio generation diffusion models (like Stable Diffusion or FLUX).

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [ ]:
!pip install --upgrade transformers diffusers datasets gradio python-dotenv huggingface_hub ipython accelerate

In [ ]:
import torch

print("--- GPU Verification ---")
print("Is CUDA available?:   ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Active GPU Device:   ", torch.cuda.get_device_name(0))
    print("VRAM Available (GB): ", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("❌ PyTorch still cannot see your GPU. We may need to re-verify the environment pathway.")

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

# 1. Handle Authentication
load_dotenv(override=True)
hf_token = os.getenv('HF_TOKEN')
if hf_token:
    login(hf_token, add_to_git_credential=True)

# 2. Load the Pipeline directly to GPU
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo", 
    torch_dtype=torch.float16, 
    variant="fp16"
)
pipe.to('cuda')

# 3. Generate the Image
prompt = "A class of students learning AI engineering in a vibrant pop-art style"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]

# 4. Display
display(image)

In [ ]:
# Restart the kernel

import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
from IPython.display import display
from diffusers import DiffusionPipeline
import torch

pipe = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, use_safetensors=True, variant="fp16")
pipe.to("cuda")

prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

image = pipe(prompt=prompt, num_inference_steps=30).images[0]

display(image)


In [ ]:
# Restart the kernel

import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
from diffusers import DiffusionPipeline
import torch

base = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
base.to("cuda")
refiner = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-refiner-1.0", text_encoder_2=base.text_encoder_2, vae=base.vae, torch_dtype=torch.float16, use_safetensors=True, variant="fp16",)
refiner.to("cuda")

# Define how many steps and what % of steps to be run on each experts (80/20) here
n_steps = 40
high_noise_frac = 0.8

prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

# run both experts
image = base(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_end=high_noise_frac,
    output_type="latent",
).images

image = refiner(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_start=high_noise_frac,
    image=image,
).images[0]

display(image)


In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
from transformers import pipeline
from datasets import load_dataset
import soundfile as sf
import torch
from IPython.display import Audio

synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device='cuda')
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
speech = synthesiser("Hi to an artificial intelligence engineer, on the way to mastery!", forward_params={"speaker_embeddings": speaker_embedding})

Audio(speech["audio"], rate=speech["sampling_rate"])

In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import torch

print("--- GPU Verification ---")
print("Is CUDA available?:   ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Active GPU Device:   ", torch.cuda.get_device_name(0))
    print("VRAM Available (GB): ", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("❌ PyTorch still cannot see your GPU. We may need to re-verify the environment pathway.")

In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
!pip install soundfile

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import soundfile as sf
from dotenv import load_dotenv
from huggingface_hub import login
from IPython.display import display, Audio
from datasets import load_dataset
print("Step 1: Basic libraries loaded successfully!")

# Let's test Torch by itself
import torch
print(f"Step 2: Torch loaded successfully! CUDA available: {torch.cuda.is_available()}")

# Let's test Transformers
from transformers import pipeline
print("Step 3: Transformers loaded successfully!")

# Let's test Diffusers
from diffusers import AutoPipelineForText2Image, DiffusionPipeline
print("Step 4: Diffusers loaded successfully!")

In [ ]:
import os
# Prevent OpenMP and memory allocation crashes on Windows
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import soundfile as sf
from transformers import pipeline
from datasets import load_dataset
from IPython.display import Audio

# 1. Clear out any residual VRAM before initializing
torch.cuda.empty_cache()

# 2. Initialize the pipeline with half-precision (float16) compression
# Pass `torch_dtype=torch.float16` to compress the model weights in VRAM
synthesiser = pipeline(
    "text-to-speech", 
    "microsoft/speecht5_tts", 
    torch_dtype=torch.float16,
    device_map="auto" # <-- The moment your VRAM is completely full, it automatically pushes the remaining layers of the model into your system RAM (CPU).
)

# 3. Load dataset and format speaker embeddings to match the float16 precision
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)

# CRITICAL: The speaker embedding tensor must be converted to .half() (float16) 
# to match the compressed pipeline, otherwise PyTorch will throw a type-mismatch error.
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0).cuda().half()

# 4. Generate speech safely
speech = synthesiser(
    "Hi to an artificial intelligence engineer, on the way to mastery!", 
    forward_params={"speaker_embeddings": speaker_embedding}
)

# 5. Play audio output
Audio(speech["audio"], rate=speech["sampling_rate"])

In [ ]:
import os
# Force sequential CUDA allocation and prevent OpenMP crashes
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from diffusers import DiffusionPipeline

# 1. Clear out any junk left over in your GPU memory
torch.cuda.empty_cache()

# 2. When you load your pipeline (example using Stable Diffusion), 
# always use float16 and offload to CPU if your GPU has less than 12GB VRAM.
model_id = "runwayml/stable-diffusion-v1-5" 

pipe = DiffusionPipeline.from_pretrained(
    model_id, 
    torch_dtype=torch.float16, # Uses half the memory of default float32
    use_safetensors=True
)

# Move to GPU safely
pipe = pipe.to("cuda")

# CRITICAL FOR WINDOWS/LOW VRAM: 
# If it still crashes here, uncomment the line below to save massive VRAM:
# pipe.enable_model_cpu_offload()

print("Pipeline loaded and ready to generate without crashing!")

In [ ]:
# 1. Handle Authentication
import os
from dotenv import load_dotenv
load_dotenv(override=True)
hf_token = os.getenv('HF_TOKEN')
if hf_token:
    login(hf_token, add_to_git_credential=True)

#### WORKNIG WITH MORE SOPHISTICATED POWERFULL MODELS, REQUIRING 100 GB CUDA SPACE

In [ ]:
import torch
from diffusers import FluxPipeline
from IPython.display import display
from datetime import datetime

start = datetime.now()

pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", torch_dtype=torch.bfloat16).to("cuda")
generator = torch.Generator(device="cuda").manual_seed(0)
prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

# Generate the image using the GPU
image = pipe(
    prompt,
    guidance_scale=0.0,
    num_inference_steps=4,
    max_sequence_length=256,
    generator=generator
).images[0]

display(image)

stop = datetime.now()

#### USING PIPELINES FROM HUGGING FACE

**Welcome to pipelines**

The HuggingFace transformers library provides APIs at two different levels.

The High Level API for using open-source models for typical inference tasks is called **pipelines**. It's incredibly easy to use.

You create a pipeline using something like:

`my_pipeline = pipeline("the_task_I_want_to_do")`

Followed by

`result = my_pipeline(my_input)`



Before we start, 2 important pro-tips for using Colab:

**Pro-tip 1:**

Data Science code often gives warnings and messages. They can mostly be safely ignored! Glance over them, and if something goes wrong later, perhaps they can give you a clue.

**Pro-tip 2:**

In the middle of running a Colab, you might get an error like this:

Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]

This is a super-misleading error message! Please don't try changing versions of packages...

This actually happens because Google has switched out your Colab runtime, perhaps because Google Colab was too busy. The solution is:

1. Kernel menu >> Disconnect and delete runtime.
2. Reload the colab from fresh and Edit menu >> Clear All Outputs.
3. Connect to a new T4 using the button at the top right.
4. Select "View resources" from the menu on the top right to confirm you have a GPU.
5. Rerun the cells in the colab, from the top down, starting with the pip installs.

**TRAINING VS INFERENCE**

1. TRAINING

**Training** is when you provide a model with data for it to adapt to get better at a task in the future. It does this by updating its internal settings - the parameters or weights of the model. If you're Training a model that's already had some training, the activity is called "fine-tuning".

2. INFERENCE

**Inference** is when you are working with a model that has _already been trained_. You are using that model to produce new outputs on new inputs, taking advantage of everything it learned while it was being trained. Inference is also sometimes referred to as "Execution" or "Running a model".

All of our use of APIs for GPT, Claude and Gemini in the last weeks are examples of **inference**. The "P" in GPT stands for "Pre-trained", meaning that it has already been trained with data (lots of it!) In week 6 we will try fine-tuning GPT ourselves.
  
The pipelines API in HuggingFace is only for use for **inference** - running a model that has already been trained. In week 7 we will be training our own model, and we will need to use the more advanced HuggingFace APIs that we look at in the up-coming lecture.


A simple way to run inference for common tasks, without worrying about all the plumbing, picking reasonable defaults.

**How it works:**

**STEP 1**: Create a pipeline - a function you can then call

```python
my_pipeline = pipeline(task, model=xx, device=xx)
```

If you don't specify a model, then Hugging Face picks one for you that's the default for the task. Specify "cuda" for the device to use an NVIDIA GPU like the one on the T4. Specify "mps" on a Mac.


**STEP 2**: Then call it as many times as you want:

```python
my_pipeline(input1)
my_pipeline(input2)
```

In [ ]:
# Importing necessary libraries for GPU verification and model loading

import torch
from huggingface_hub import login
from transformers import pipeline
print("Imported torch, huggingface_hub, and transformers successfully!")


In [ ]:
from diffusers import DiffusionPipeline
print("Imported diffusion pipeline library successfully!")

In [ ]:
import soundfile as sf
from IPython.display import Audio
print("Imported soundfile and IPython.display successfully!")

In [ ]:
# 1. Handle Authentication
import os
from dotenv import load_dotenv
load_dotenv(override=True)
hf_token = os.getenv('HF_TOKEN')
if hf_token:
    login(hf_token, add_to_git_credential=True)

In [ ]:
# Sentiment Analysis Pipeline Test
my_simple_sentiment_analyzer = pipeline("sentiment-analysis", device="cuda")
result = my_simple_sentiment_analyzer("I'm super excited to be on the way to LLM mastery!")
print(result)

In [ ]:
result = my_simple_sentiment_analyzer("I should be more excited to be on the way to LLM mastery!")
print(result)

In [ ]:
better_sentiment = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device="cuda")
result = better_sentiment("I should be more excited to be on the way to LLM mastery!!")
print(result)

In [ ]:
# Named Entity Recognition

ner = pipeline("ner", device="cuda")
result = ner("AI Engineers are learning about the amazing pipelines from HuggingFace in Google Colab from Ed Donner")
for entity in result:
  print(entity)

#### NER VS SENTIMENT ANALYSIS
While both Named Entity Recognition (NER) and Sentiment Analysis fall under the umbrella of Natural Language Processing (NLP), they serve completely different purposes. Think of NER as the "Who, What, and Where" detector, while Sentiment Analysis is the "How do they feel" detector.

__The Core Difference__

* __Named Entity Recognition (NER)__ is an information extraction task. It identifies specific real-world objects (entities) within a text and classifies them into predefined categories like names of people, organizations, locations, dates, or product codes.
* __Sentiment Analysis__ is a text classification task. It determines the emotional tone behind a body of text, typically categorizing it as positive, negative, or neutral (and sometimes extracting specific emotions like anger, joy, or frustration).

__How They Work Together__
In real-world AI applications, these two techniques are rarely used in isolation; they are often chained together to unlock deeper insights. This is called __Aspect-Based Sentiment Analysis (ABSA)__.

For instance, if a company scans thousands of customer reviews, NER identifies what product or store location is being talked about, and Sentiment Analysis tells them whether people like it.

Example: "The food at McDonalds [Entity] was fantastic [Positive], but the service was terrible [Negative]."

Without NER, you'd just know the review was mixed. With both, you know exactly what parts of the business need work.

__A Side-by-Side Comparison__
Imagine processing this customer tweet:
_"I just bought the new iPhone from the Apple Store in New York, and it is absolutely incredible!"_

Here is how both systems analyze the exact same sentence:

__Feature__ __| Named Entity Recognition (NER) | Sentiment Analysis__

_Primary Goal_  |  Identify and label specific "nouns" (entities) | Identify the overall emotional undertone

_What it extracts_ | iPhone $\rightarrow$ Product • Apple Store $\rightarrow$ Organization • New York $\rightarrow$ Location | absolutely incredible $\rightarrow$ Positive Sentiment (High confidence)

_Output Type_ | Token-level tags (labels mapped to specific words) | Text-level or sentence-level score/class

_Common Algorithms_ | Conditional Random Fields (CRFs), BERT token classification | Logistic Regression, Finetuned LLMs, VADER.

In [ ]:
# Question Answering with Context

# 1. Initialize a text generation pipeline (using a lightweight instruction model like Llama/Qwen/Gemma)
# If you don't specify a model, transformers will pick a default one.
qa_generator = pipeline("text-generation", device="cuda")

question = "What are Hugging Face pipelines?"
context = "Pipelines are a high level API for inference of LLMs with common tasks"

# 2. Structure your prompt
prompt = f"Context: {context}\n\nQuestion: {question}\n\nAnswer:"

# 3. Generate the response
result = qa_generator(prompt, max_new_tokens=30, clean_up_tokenization_spaces=True)
print(result[0]['generated_text'])

__AVAILABLE TASKS FOR PIPELINES__
 
 ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']

#### SKIPPING THE PIPELINE AND DOING NATIVELY

In [ ]:
# Text Summarization

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

# 1. Load the model and tokenizer
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda")

# 2. Your text payload
text = """
The Hugging Face transformers library is an incredibly versatile and powerful tool for natural language processing (NLP).
It allows users to perform a wide range of tasks such as text classification, named entity recognition, and question answering, among others.
It's an extremely popular library that's widely used by the open-source data science community.
It lowers the barrier to entry into the field by providing Data Scientists with a productive, convenient way to work with transformer models.
"""

# 3. Native Processing (Bypassing pipeline entirely)
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
summary_ids = model.generate(inputs["input_ids"], max_length=50, min_length=25, do_sample=False)
summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("--- Generated Summary ---")
print(summary_text)

__What is happening behind the scenes here?__

Instead of relying on the high-level pipeline wrapper to manage the steps, you are managing the standard sequence-to-sequence workflow yourself:

* `tokenizer(...)` takes your raw string, truncates it to fit the model's max context limit, maps the words to numeric token IDs, and sends the tensors straight to your GPU (.to("cuda")).

* `model.generate(...)` executes the encoder-decoder attention math directly on your RTX 5060, processing your constraints (max_length and min_length) natively.

* `tokenizer.decode(...)` translates the generated output array of numerical IDs back into human-readable English text, while cleaning up structural formatting symbols (skip_special_tokens=True).

The latest major release of __transformers (v5)__, which completely removed the legacy, hardcoded "translation_en_to_fr" string task from the pipeline lookup registry to favor modern multi-task models.

Because your absolute constraint is ONLY USING PIPELINE, you can resolve this by doing one of two things: use the standard __"text-generation"__ pipeline task with a prompt, or explicitly supply a __translation model to the "text-classification" or "any-to-any"__ architecture pipeline configuration.

The cleanest, industry-standard way to solve this in v5 using only `pipeline` is to route it through `text-generation`.

__Solution 1: Use pipeline("text-generation") (Recommended for v5)__

You can use the pipeline you already downloaded (`SmollM3-3B`) or let it default to a text generation model. You simply structure the text as a translation instruction.


In [ ]:
from transformers import pipeline

# Use the active text-generation pipeline task
translator = pipeline("text-generation", device="cuda")

text_to_translate = "The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API."

# Structure a prompt the model understands
prompt = f"Translate this English text to French:\n\"{text_to_translate}\"\n\nFrench Translation:"

result = translator(prompt, max_new_tokens=60, clean_up_tokenization_spaces=True)
print(result[0]['generated_text'])

__Solution 2: Explicitly load a translation model into pipeline__

If you want to use a classic translation model like `Helsinki-NLP/opus-mt-en-fr` within a pipeline context without running into registry string lookup blockers, you can pass the model explicitly while using the broad "`text-generation`" or fallback generation pipeline wrapper:

In [ ]:
from transformers import pipeline, AutoModelForSeq2SeqLM
from transformers.models.marian import MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-en-fr"

# 1. Load the explicit model and tokenizer
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = MarianTokenizer.from_pretrained(model_name)

# 2. Use 'text-classification' as a generic text pipeline wrapper
translator = pipeline("text-classification", model=model, tokenizer=tokenizer, device="cuda")

# 3. Translate
text = "The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API."
result = translator(text)

print("--- French Translation ---")
print(result[0]['generated_text'])

In [ ]:
# Classification

classifier = pipeline("zero-shot-classification", device="cuda")
result = classifier("Hugging Face's Transformers library is amazing!", candidate_labels=["technology", "sports", "politics"])
print(result)

In [ ]:
# Text Generation

generator = pipeline("text-generation", device="cuda")
result = generator("If there's one thing I want you to remember about using HuggingFace pipelines, it's")
print(result[0]['generated_text'])

In [ ]:
# Image Generation - remember this?! Now you know what's going on
# Pipelines can be used for diffusion models as well as transformers

from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
pipe.to("cuda")
prompt = "A class of students learning AI engineering in a vibrant pop-art style"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
display(image)

In [ ]:
# Audio Generation

from transformers import pipeline
from datasets import load_dataset
import soundfile as sf
import torch
from IPython.display import Audio

synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device='cuda')
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
speech = synthesiser("Hi to an artificial intelligence engineer, on the way to mastery!", forward_params={"speaker_embeddings": speaker_embedding})

Audio(speech["audio"], rate=speech["sampling_rate"])

1. __Importing the Tools__

`from transformers import pipeline`

`from datasets import load_dataset`

`import soundfile as sf`

`import torch`

`from IPython.display import Audio`

* __pipeline:__ The high-level Hugging Face wrapper used to easily load and infer deep learning models.

* __load_dataset:__ A utility from the Hugging Face datasets library used to download public datasets (in this case, containing voice profiles).

* __torch:__ PyTorch—the backend math engine running the tensor calculations on your GPU.

* __Audio:__ An IPython module that embeds a native audio player widget directly inside your notebook cells.

2. __Initializing the TTS Engine__

`synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device='cuda')`

This line builds the generation engine.

* It explicitly requests the `"text-to-speech"` task.

* It downloads `microsoft/speecht5_tts`, a lightweight transformer-based speech model.

* `device='cuda':` This forces the pipeline to load the neural network weights into your graphics card's VRAM, accelerating the sound synthesis process.

3. __Loading a Voice Profile (Speaker Embedding)__

`embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)`

`speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)`

The `SpeechT5` architecture is __voice-agnostic__—it needs to be told how to sound (e.g., a high-pitched voice, a deep baritone, an accent).

* __load_dataset(...):__ Fetches a dataset containing pre-calculated X-vectors. An X-vector is a mathematical profile (embedding) representing the unique vocal characteristics of a human speaker.

* __embeddings_dataset[7306]:__ Pulls a specific speaker profile from row 7306 of the dataset.

* __.unsqueeze(0):__ Changes the shape of the array to include a batch dimension (turning it from a flat array into a 2D matrix), which PyTorch models strictly require for structural input compatibility.

4. __Synthesizing the Audio__

`speech = synthesiser("Hi to an artificial intelligence engineer, on the way to mastery!", forward_params={"speaker_embeddings": speaker_embedding})`

This line triggers the actual generation block.

* Your text string is converted into tokens.

* `forward_params:` Passes the speaker_embedding matrix directly to the model's core layers, conditioning the AI to clone that specific speaker's tone, pacing, and pitch.

* The output `(speech)` is returned as a Python dictionary containing two critical keys: the raw synthesized audio waveform array and the target sampling rate.

5. __Rendering the Audio Player__

`Audio(speech["audio"], rate=speech["sampling_rate"])`

This feeds the raw numerical array (`speech["audio"]`) and the frequency mapping configuration (`speech["sampling_rate"]`, which is typically 16,000Hz for this model) into the IPython interpreter.

It draws a playable bar containing Play/Pause, Timeline, and Volume sliders right at the bottom of your cell output.


__How the Model Processes Your Text under the Hood__
When this cell runs, `SpeechT5` executes a multi-stage transformer pipeline:

1. __Text Encoder:__ Transforms your text prompt into an array of linguistic representations.

2. __Speech Decoder:__ Combines those text representations with the speaker_embedding to compute a sequence of mel-spectrograms (visual representations of frequencies over time).

3. __Vocoder (HiFi-GAN):__ Converts those calculated frequency frames back into a raw, audible audio waveform.

### TOKENIZERS

In [ ]:
from transformers import AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B', trust_remote_code=True)

In [ ]:
text = "I am excited to show Tokenizers in action to my LLM engineers"
tokens = tokenizer.encode(text)
tokens

In [ ]:
character_count = len(text)
word_count = len(text.split(' '))
token_count = len(tokens)
print(f"There are {character_count} characters, {word_count} words and {token_count} tokens")

In [ ]:
tokenizer.decode(tokens)

In [ ]:
tokenizer.batch_decode(tokens)

In [ ]:
# tokenizer.vocab
tokenizer.get_added_vocab()

In [ ]:
len(tokenizer.vocab)

### Instruct variants of models

Many models have a variant that has been trained for use in Chats.  
These are typically labelled with the word "Instruct" at the end.  
They have been trained to expect prompts with a particular format that includes system, user and assistant prompts.  

There is a utility method `apply_chat_template` that will convert from the messages list format we are familiar with, into the right input prompt for this model.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B-Instruct', trust_remote_code=True)

In [ ]:

messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)

### Crucial "Aha" moment

For 2.5 weeks, I've given you the impression that LLMs could receive a list of python dictionaries in some way:

```python
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]
```

But an LLM is just a Data Science model that takes a sequence of numbers and predicts the probability of the next number! You can't pass a bunch of Python objects into a statistical model!

#### And now you have the missing piece of the puzzle..

The messages in OpenAI format get converted:

1. ...into a sequence of words with special tags to separate the System, User, Assistant prompt
2. ...then the words are broken down into fragments - "tokens"
3. ...then the tokens are replaced with Token IDs - and this is the input sequence

> The input to an LLM is a sequence of Token IDs. The output is the probability distribution of the next Token ID to follow this input.

That's it!


### Trying new models

We will now work with 3 models:

* __Phi4 from Microsoft__  
* __DeepSeek 3.1 from DeepSeek AI__  
* __QwenCoder 2.5 from Alibaba Cloud__

In [ ]:
PHI4 = "microsoft/Phi-4-mini-instruct"
DEEPSEEK = "deepseek-ai/DeepSeek-V3.1"
QWEN_CODER = "Qwen/Qwen2.5-Coder-7B-Instruct"

In [ ]:
phi4_tokenizer = AutoTokenizer.from_pretrained(PHI4)

text = "I am curiously excited to show Hugging Face Tokenizers in action to my LLM engineers"
print("Llama:")
tokens = tokenizer.encode(text)
print(tokens)
print(tokenizer.batch_decode(tokens))
print("\nPhi 4:")
tokens = phi4_tokenizer.encode(text)
print(tokens)
print(phi4_tokenizer.batch_decode(tokens))

In [ ]:
print("Llama:")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nPhi 4:")
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

Llama:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

Tell a light-hearted joke for a room of Data Scientists<|eot_id|><|start_header_id|>assistant<|end_header_id|>



Phi 4:
<|system|>You are a helpful assistant<|end|><|user|>Tell a light-hearted joke for a room of Data Scientists<|end|><|assistant|>

In [ ]:
deepseek_tokenizer = AutoTokenizer.from_pretrained(DEEPSEEK)

text = "I am curiously excited to show Hugging Face Tokenizers in action to my LLM engineers"
print(tokenizer.encode(text))
print()
print(phi4_tokenizer.encode(text))
print()
print(deepseek_tokenizer.encode(text))

[128000, 40, 1097, 2917, 13610, 12304, 311, 1501, 473, 36368, 19109, 9857, 12509, 304, 1957, 311, 856, 445, 11237, 25175]

[40, 939, 4396, 23138, 15209, 316, 2356, 59116, 4512, 29049, 17951, 24223, 306, 3736, 316, 922, 451, 19641, 32437]

[0, 43, 1030, 108771, 15046, 304, 1801, 24133, 5426, 11906, 47948, 24524, 295, 4271, 304, 1026, 33792, 47, 26170]

In [ ]:
print("Llama:")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nPhi:")
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nDeepSeek:")
print(deepseek_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

Llama:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

Tell a light-hearted joke for a room of Data Scientists<|eot_id|><|start_header_id|>assistant<|end_header_id|>



Phi:
<|system|>You are a helpful assistant<|end|><|user|>Tell a light-hearted joke for a room of Data Scientists<|end|><|assistant|>

DeepSeek:
<｜begin▁of▁sentence｜>You are a helpful assistant<｜User｜>Tell a light-hearted joke for a room of Data Scientists<｜Assistant｜></think>

In [ ]:
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_CODER)
code = """
def hello_world(person):
  print("Hello", person)
"""
tokens = qwen_tokenizer.encode(code)
for token in tokens:
  print(f"{token}={qwen_tokenizer.decode(token)}")

198=

750=def
23811= hello
31792=_world
29766=(person
982=):

220= 
1173= print
445=("
9707=Hello
497=",
1697= person
340=)

### Accessing Llama

Yesterday you should have received approval to use this model:

https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct

You can either use that today, or it's faster if you get approval for this model too.

https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

Select this link to see if you need to request approval too. Pick the version of Llama that you want below by commenting out one of these! Or skip Llama altogether.

In [ ]:
# instruct models and 1 reasoning model

# Llama 3.1 is larger and you should already be approved
# see here: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct

# LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Llama 3.2 is smaller but you might need to request access again
# see here: https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

LLAMA = "meta-llama/Llama-3.2-1B-Instruct"

PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [ ]:
messages = [
    {"role": "user", "content": "Tell a joke for a room of Data Scientists"}
  ]

### Accessing Llama 3.1 from Meta

In order to use the fantastic Llama 3.1, Meta does require you to sign their terms of service.

Visit their model instructions page in Hugging Face: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B

At the top of the page are instructions on how to agree to their terms. If possible, you should use the same email as your huggingface account.

In my experience approval comes in a couple of minutes. Once you've been approved for any 3.1 model, it applies to the whole family of models.

If you have any problems accessing Llama, please see this colab, including some suggestions if you don't get approved by Meta for any reason.

https://colab.research.google.com/drive/1deJO03YZTXUwcq2vzxWbiBhrRuI29Vo8

In [ ]:
# Quantization Config - this allows us to load the model into memory and use less memory

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
# Tokenizer

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

In [ ]:
inputs

tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,    777,  12044,    220,   2366,     21,    271, 128009, 128006,
            882, 128007,    271,  41551,    264,  22380,    369,    264,   3130,
            315,   2956,  57116, 128009]], device='cuda:0')

In [ ]:
# The model

model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)

In [ ]:
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:,.1f} MB")

Memory footprint: 1,012.0 MB

### Looking under the hood at the Transformer model

The next cell prints the HuggingFace `model` object for Llama.

This model object is a Neural Network, implemented with the Python framework PyTorch. The Neural Network uses the architecture invented by Google scientists in 2017: the Transformer architecture.

While we're not going to go deep into the theory, this is an opportunity to get some intuition for what the Transformer actually is.

If you're completely new to Neural Networks, check out my [YouTube intro playlist](https://www.youtube.com/playlist?list=PLWHe-9GP9SMMdl6SLaovUQF2abiLGbMjs) for the foundations.

Now take a look at the layers of the Neural Network that get printed in the next cell. Look out for this:

- It consists of layers
- There's something called "embedding" - this takes tokens and turns them into 4,096 dimensional vectors. We'll learn more about this in Week 5.
- There are then 16 sets of groups of layers (32 for Llama 3.1) called "Decoder layers". Each Decoder layer contains three types of layer: (a) self-attention layers (b) multi-layer perceptron (MLP) layers (c) batch norm layers.
- There is an LM Head layer at the end; this produces the output

Notice the mention that the model has been quantized to 4 bits.

It's not required to go any deeper into the theory at this point, but if you'd like to, I've asked our mutual friend to take this printout and make a tutorial to walk through each layer. This also looks at the dimensions at each point. If you're interested, work through this tutorial after running the next cell:

https://chatgpt.com/canvas/shared/680cbea6de688191a20f350a2293c76b

In [ ]:
# Execute this cell and look at what gets printed; investigate the layers

model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb): LlamaRotaryEmbedding()
  )
  (lm_head): Linear(in_features=2048, out_features=128256, bias=False)
)

### And if you want to go even deeper into Transformers

In addition to looking at each of the layers in the model, you can actually look at the HuggingFace code that implements Llama using PyTorch.

Here is the HuggingFace Transformers repo:  
https://github.com/huggingface/transformers

And within this, here is the code for Llama 4:  
https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama4/modeling_llama4.py

Obviously it's not neceesary at all to get into this detail - the job of an AI engineer is to select, optimize, fine-tune and apply LLMs rather than to code a transformer in PyTorch. OpenAI, Meta and other frontier labs spent millions building and training these models. But it's a fascinating rabbit hole if you're interested!

In [ ]:
# OK, with that, now let's run the model!

outputs = model.generate(inputs, max_new_tokens=80)
outputs[0]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
tensor([128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
            25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
           220,    777,  12044,    220,   2366,     21,    271, 128009, 128006,
           882, 128007,    271,  41551,    264,  22380,    369,    264,   3130,
           315,   2956,  57116, 128009, 128006,  78191, 128007,    271,   8586,
           596,    264,  22380,  41891,    311,    264,   3130,    315,    828,
         14248,   1473,  77955,   1550,    279,  10550,    733,    311,  15419,
            30,   9393,    433,    574,   8430,    264,   2697,    364,   4338,
         10767,      6,    323,   4460,    311,    364,  31690,      6,   1202,
         21958,     13,   2030,    304,    279,    842,     11,    433,   1120,
          4460,    311,    364,  12727,      6,    709,   1202,    659,  66104,
          2136,   2266,   2028,  22380,  11335,    389,    279,  11156,   3878,
          1511,    304,    828,   8198,     11,   1778,    439,    330,  39560,
             1,    323,    330], device='cuda:0')

In [ ]:
# Well that doesn't make much sense!
# How about this..

tokenizer.decode(outputs[0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 19 Jun 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

Tell a joke for a room of Data Scientists<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Here's a joke tailored to a room of data scientists:

"Why did the dataset go to therapy? Because it was feeling a little 'disordered' and needed to 'normalize' its emotions. But in the end, it just needed to'scale' up its self-awareness."

This joke plays on the technical terms used in data science, such as "normalized" and "

In [ ]:
# Clean up memory
# Thank you Kuan L. for helping me get this to properly free up memory!
# If you select "Show Resources" on the top right to see GPU memory, it might not drop down right away
# But it does seem that the memory is available for use by new models in the later code.

del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()

### A couple of quick notes on the next block of code:

I'm using a HuggingFace utility called TextStreamer so that results stream back.
To stream results, we simply replace:  
`outputs = model.generate(inputs, max_new_tokens=80)`  
With:  
`streamer = TextStreamer(tokenizer)`  
`outputs = model.generate(inputs, max_new_tokens=80, streamer=streamer)`

Also I've added the argument `add_generation_prompt=True` to my call to create the Chat template. This ensures that Phi generates a response to the question, instead of just predicting how the user prompt continues. Try experimenting with setting this to False to see what happens. You can read about this argument here:

https://huggingface.co/docs/transformers/main/en/chat_templating#what-are-generation-prompts

Thank you to student Piotr B for raising the issue!

In [ ]:
# Wrapping everything in a function - and adding Streaming and generation prompts

def generate(model, messages, quant=True, max_new_tokens=80):
  tokenizer = AutoTokenizer.from_pretrained(model)
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
  attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")
  streamer = TextStreamer(tokenizer)
  if quant:
    model = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant_config).to("cuda")
  else:
    model = AutoModelForCausalLM.from_pretrained(model).to("cuda")
  outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)

In [ ]:
generate(PHI, messages)

<|user|>Tell a joke for a room of Data Scientists<|end|><|assistant|>Sure, here's a joke tailored for data scientists:

Why did the data scientist break up with the computer?

Because it kept dumping data on them, and they just couldn't handle the relationship anymore!<|end|>

### Accessing Gemma from Google

A student let me know (thank you, Alex K!) that Google also now requires you to accept their terms in HuggingFace before you use Gemma.

Please visit their model page at this link and confirm you're OK with their terms, so that you're granted access.

https://huggingface.co/google/gemma-3-270m-it

In [ ]:
messages = [
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]
generate(GEMMA, messages, quant=False)

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
<bos><start_of_turn>user
Tell a light-hearted joke for a room of Data Scientists<end_of_turn>
<start_of_turn>model
Why did the data scientist get fired from his job? 

Because he kept making the wrong numbers!
<end_of_turn>

In [ ]:
generate(QWEN, messages)

<|im_start|>user
Tell a light-hearted joke for a room of Data Scientists<|im_end|>
<|im_start|>assistant
Sure! Here's a light-hearted joke *specifically* tailored for a room of Data Scientists:

---

Why did the data scientist break up with their statistician?

Because they just couldn’t handle the *correlation* — it was always *r*elated, but never *significant* enough to make a lasting relationship! 😂

*(Bonus points if you know the joke is a play)

In [ ]:
generate(DEEPSEEK, messages, quant=False, max_new_tokens=500)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
<｜begin▁of▁sentence｜><｜User｜>Tell a light-hearted joke for a room of Data Scientists<｜Assistant｜><think>
Alright, so the user wants a light-hearted joke for a room of Data Scientists. Hmm, okay. Let me break this down.

First, I need to think about what Data Scientists are passionate about. They deal with numbers, data, and maybe even some quirky stuff. So, the joke needs to be light-hearted but still relevant to their field.

Maybe something about their work. They're often seen working with data, maybe even with numbers that are a bit annoying. Like, how about something with a cat or a dog? That could be a good metaphor.

Wait, the user mentioned a room of Data Scientists, so maybe something that's universally relatable, like a joke that everyone can get on with. Maybe something that's a bit of a joke about their work.

Let me think of a common phrase. "How do you keep a dog happy?" That's classic. But how to tie it into Data Science? Maybe something like, "How do you keep a dog happy? By working with numbers!" That sounds a bit odd but fits the theme.

Alternatively, maybe "How do you keep a dog happy? By working with data!" That's a bit more straightforward but still funny. It plays on the idea of working with data, which is a key part of their job.

Another angle could be using a pun, but I think the first one is more of a joke than a pun. It's a play on the word "happy" and "working" with numbers or data. It's simple and gets across the idea that they're serious about their work but in a light-hearted way.

I should also consider if the user wants something more specific, but since they just asked for a joke, this should work. It's simple, uses a common phrase, and relates to their field in a way that's relatable. I think that's a good balance between being light-hearted and relevant.
</think>

How do you keep a dog happy? By working with numbers! 🐾<｜end▁of▁sentence｜>